<a href="https://colab.research.google.com/github/aravindchandra/Aravind_INFO5731_Spring2026/blob/main/In_class_exercises_4_Text_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Data Preprocessing & Cleaning (Text)  
**Time:** 20 minutes  |  **Points:** 10  

## Instructions
- This is an individual in-class assignment.  
- Write your code **inside each answer cell**.  
- Print the required outputs.  
- Submit your GitHub/Colab link as instructed by the instructor.


You are given a small dataset of customer support messages as a **TAB-separated text file**:  
- `support_messages.txt`

You will download this file from **Canvas** and upload it to your **Google Colab** notebook.

**How to upload it to your Google Colab notebook?**

1. Download `support_messages.txt` from Canvas.
3. In **the left sidebar**, click the **Files** icon (folder).  
4. Click **Upload** and select `support_messages.txt`.

6. RightAfter uploading, the file will appear in the Colab file list on the left.

6. Right-click the file, copy its path, and paste it into the FILE_PATH variable in Q1.

7. Run Q1 to load the dataset.



> Important: Keep the file name exactly as `support_messages.txt`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Questions (Total = 10 points)

### Q1 (1 point) — Load the dataset
Load the TAB-separated file into a pandas DataFrame with columns: `id`, `message`.  
Print: **(a)** `df.shape`, **(b)** `df.head(3)`.

### Q2 (3 points) — Descriptive columns
Add these columns for each message and print the full DataFrame:
- `word_count`: number of words  
- `char_count`: number of characters  
- `num_count`: number of digits (0–9)  
- `upper_word_count`: number of ALL-CAPS words (e.g., `"WHY"`, `"DAMAGED"`)  

### Q3 (3 points) — Clean text
Build a `clean_text(text)` function and create a new column `clean` with these steps **in order**:
1) lowercase  
2) remove punctuation/symbols (keep letters/numbers/spaces)  
3) remove English stopwords (use **nltk** or **sklearn** list)  
4) remove extra spaces  

Print the **original** message and **clean** version for rows `id=1` and `id=4`.

### Q4 (2 points) — Regex extraction
Using RegEx, extract and create two new columns:
- `order_id`: first occurrence of pattern `ORD-####` (case-insensitive; `ord-1060` is valid)  
- `email`: first email address if present (otherwise `None`/`NaN`)  

Print: `id`, `order_id`, `email` for all rows.

### Q5 (1 point) — TF-IDF keywords
Using the `clean` column, compute **TF-IDF** for the messages and print the **top 5 keywords** with the highest **average TF-IDF** across documents.


In [ ]:
# Setup (run this cell first)
import re
import pandas as pd


## Q1 (1 point) — Answer below

In [11]:
# Q1 — ANSWER CELL
FILE_PATH = "/content/support_messages.txt"

# TODO: load the TAB-separated file into df
# Hint: pd.read_csv(FILE_PATH, sep="\t")
df = None

# TODO: print df.shape and df.head(3)

df = pd.read_csv("/content/support_messages.txt", sep="\t", names=["id", "message"])

# Print shape and first 3 rows
print("Q1 Output:")

print(df.shape)
print(df.head(3))


Q1 Output:
(9, 2)
   id                                            message
0  id                                            message
1   1  Hi!! My ORDER is late :(  Order# ORD-1042. Ema...
2   2  Refund please!!! I was charged 2 times... invo...


## Q2 (3 points) — Answer below

In [10]:
# Q2 — ANSWER CELL
# TODO: create word_count, char_count, num_count, upper_word_count
# Hint for digits: df["message"].str.count(r"\d")
# Hint for ALL-CAPS words: count tokens where token.isupper()

# TODO: display/print the full DataFrame

def count_upper_words(text):
    return sum(1 for word in text.split() if word.isupper())

df["word_count"] = df["message"].apply(lambda x: len(x.split()))
df["char_count"] = df["message"].apply(len)
df["num_count"] = df["message"].apply(lambda x: sum(c.isdigit() for c in x))
df["upper_word_count"] = df["message"].apply(count_upper_words)

print("\nQ2 Output:")
print(df)



Q2 Output:
   id                                            message  word_count  \
0  id                                            message           1   
1   1  Hi!! My ORDER is late :(  Order# ORD-1042. Ema...          12   
2   2  Refund please!!! I was charged 2 times... invo...          11   
3   3        Great service, thanks! arrived in 2 days :)           8   
4   4  WHY is my package DAMAGED??? tracking says del...           8   
5   5  Need to change address: 7421 Frankford Rd Apt ...          12   
6   6  Support ticket: ORD-1050. Call me at (469) 555...           9   
7   7  I can’t login— password reset link not working...          10   
8   8  Item missing from box. pls send replacement!! ...          10   

   char_count  num_count  upper_word_count  
0           7          0                 0  
1          73          4                 2  
2          71          9                 3  
3          43          1                 0  
4          55          0                 2

## Q3 (3 points) — Answer below

In [24]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # keep letters/numbers/spaces
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return re.sub(r"\s+", " ", " ".join(tokens)).strip()

df["clean"] = df["message"].apply(clean_text)

# Print original + clean for id=1 and id=4
for target_id in [1, 4]:
    # Fix: Convert target_id to string for comparison
    row = df[df["id"] == str(target_id)].iloc[0]
    print("ID:", row["id"])
    print("ORIGINAL:", row["message"])
    print("CLEAN:   ", row["clean"])
    print("-" * 80)

ID: 1
ORIGINAL: Hi!! My ORDER is late :(  Order# ORD-1042. Email me at sara.Ali@gmail.com
CLEAN:    hi order late order ord 1042 email sara ali gmail com
--------------------------------------------------------------------------------
ID: 4
ORIGINAL: WHY is my package DAMAGED??? tracking says delivered...
CLEAN:    package damaged tracking says delivered
--------------------------------------------------------------------------------


## Q4 (2 points) — Answer below

In [15]:
# Q4 — ANSWER CELL
# order_id pattern: r"ORD-\d{4}" with re.IGNORECASE
# email pattern (simple): r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

# TODO: create df["order_id"] and df["email"]
# TODO: print/display df[["id", "order_id", "email"]]

order_pattern = re.compile(r"(ord-\d{4})", re.IGNORECASE)
email_pattern = re.compile(r"([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})")

df["order_id"] = df["message"].apply(
    lambda x: order_pattern.search(x).group(1) if order_pattern.search(x) else None
)

df["email"] = df["message"].apply(
    lambda x: email_pattern.search(x).group(1) if email_pattern.search(x) else None
)

print("\nQ4 Output:")
print(df[["id", "order_id", "email"]])



Q4 Output:
   id  order_id                  email
0  id      None                   None
1   1  ORD-1042     sara.Ali@gmail.com
2   2  ORD-1042                   None
3   3      None                   None
4   4      None                   None
5   5      None                   None
6   6  ORD-1050                   None
7   7      None  mehri.sattari@unt.edu
8   8  ord-1060                   None


## Q5 (1 point) — Answer below

In [16]:
# Q5 — ANSWER CELL
# Hint: from sklearn.feature_extraction.text import TfidfVectorizer
# 1) fit TF-IDF on df["clean"]
# 2) compute average TF-IDF per term across documents
# 3) print top 5 terms + their average scores

# TODO

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["clean"])

# Average TF-IDF score per term
avg_tfidf = tfidf_matrix.mean(axis=0).A1
terms = vectorizer.get_feature_names_out()

tfidf_scores = pd.DataFrame({
    "term": terms,
    "avg_tfidf": avg_tfidf
})

top_5 = tfidf_scores.sort_values(by="avg_tfidf", ascending=False).head(5)

print("\nQ5 Output:")
print(top_5)



Q5 Output:
       term  avg_tfidf
39      ord   0.125373
36  message   0.111111
40    order   0.088427
1      1042   0.063192
24    email   0.058799


## Grading Checklist
- Q1: correct load + prints  
- Q2: correct counts  
- Q3: cleaning follows the required order + prints for id=1 and id=4  
- Q4: regex extraction works (case-insensitive `ORD-####` and emails)  
- Q5: prints 5 keywords + their scores (rounding is fine)
